# FraudLens
## AI-Assisted Financial Statement Fraud Detection Pipeline

This notebook detects potential financial statement fraud using:
- Beneish M-Score ratios
- Profitability and leverage ratios
- Z-score anomaly detection
- Isolation Forest
- Logistic Regression risk scoring

**Author:** Theophilus Abayayateye  
**Project:** FraudLens Portfolio Project

In [1]:
# Step 1: Import all libraries FraudLens will need

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("All libraries loaded successfully!")


All libraries loaded successfully!


## Step 2: Generate Synthetic Financial Dataset

We create 100 fictional companies with two years of financial data.
Some companies will have manipulated figures (fraud = 1), others will be clean (fraud = 0).

In [3]:
# Set a random seed so we get the same numbers every time we run the code
np.random.seed(42)

# Number of companies to simulate
n = 100

# Generate clean company data (year 1)
data = pd.DataFrame({
    'company_id': range(1, n + 1),
    'sales_t' : np.random.uniform(500000, 500000, n),
    'sales_t1': np.random.uniform(500000, 500000, n),
    'receivables_t' : np.random.uniform(500000, 500000, n),
    'receivables_t1' : np.random.uniform(500000, 500000, n),
    'COGS_t' : np.random.uniform(300000, 300000, n),
    'COGS_t1' : np.random.uniform(300000, 300000, n),
    'total_assets_t' : np.random.uniform(1000000, 1000000, n),
    'total_assets_t1' : np.random.uniform(1000000, 1000000, n),
    'PPE_t' : np.random.uniform(100000, 100000, n),
    'PPE_t1' : np.random.uniform(100000, 100000, n),
    'net_income_t' : np.random.uniform(50000, 500000, n),
    'total_debt_t' : np.random.uniform(100000, 2000000, n),
    'cash_flow_ops_t' : np.random.uniform(50000, 500000, n),
})

# Label 20 compnaies as fraudulent
data ['fraud'] = 0
data.loc[data['company_id'] <= 20, 'fraud'] = 1

print(data.shape)
print(data.head())
print(data['fraud'].value_counts())




(100, 15)
   company_id   sales_t  sales_t1  receivables_t  receivables_t1    COGS_t  \
0           1  500000.0  500000.0       500000.0        500000.0  300000.0   
1           2  500000.0  500000.0       500000.0        500000.0  300000.0   
2           3  500000.0  500000.0       500000.0        500000.0  300000.0   
3           4  500000.0  500000.0       500000.0        500000.0  300000.0   
4           5  500000.0  500000.0       500000.0        500000.0  300000.0   

    COGS_t1  total_assets_t  total_assets_t1     PPE_t    PPE_t1  \
0  300000.0       1000000.0        1000000.0  100000.0  100000.0   
1  300000.0       1000000.0        1000000.0  100000.0  100000.0   
2  300000.0       1000000.0        1000000.0  100000.0  100000.0   
3  300000.0       1000000.0        1000000.0  100000.0  100000.0   
4  300000.0       1000000.0        1000000.0  100000.0  100000.0   

    net_income_t  total_debt_t  cash_flow_ops_t  fraud  
0  133309.817977  2.099011e+05    391218.438168      1 

## Step 3: Calculate Beneish M-Score Ratios

The Beneish M-Score uses 8 indices to detect earnings manipulation.
Each ratio measures a different aspect of financial statement change.

In [4]:
# Beneish Index 1: DSRI - Days Sales Receivable Index
# Measures whether receivables grew faster than sales (a red flag)
data['DSRI'] = (data['receivables_t'] / data['sales_t']) / (data['receivables_t1'] / data['sales_t1'])

# Beneish Index 2: GMI - Gross Margin Index
# Measures whether gross margin is deteriorating
data['GMI'] = ((data['sales_t1'] - data['COGS_t1']) / data['sales_t1']) / ((data['sales_t'] - data['COGS_t']) / data['sales_t'])

# Beneish Index 3: AQI - Asset Quality Index
# Measures whether non-productive assets are increasing
data['AQI'] = (1 - (data['PPE_t'] + data['receivables_t']) / data['total_assets_t']) / (1 - (data['PPE_t1'] + data['receivables_t1']) / data['total_assets_t1'])

# Beneish Index 4: SGI - Sales Growth Index
# Measures sales growth (high growth companies manipulate more)
data['SGI'] = data['sales_t'] / data['sales_t1']

print(data[['company_id', 'DSRI', 'GMI', 'AQI', 'SGI']].head())

   company_id  DSRI  GMI  AQI  SGI
0           1   1.0  1.0  1.0  1.0
1           2   1.0  1.0  1.0  1.0
2           3   1.0  1.0  1.0  1.0
3           4   1.0  1.0  1.0  1.0
4           5   1.0  1.0  1.0  1.0


In [7]:
# Make fraudulent companies look suspicious
fraud_idx = data['fraud'] == 1

# Inflate receivables for fraud companies (red flag)
data.loc[fraud_idx, 'receivables_t'] = data.loc[fraud_idx, 'receivables_t'] * 2.5

# Shrink sales for fraud companies (deteriorating performance)
data.loc[fraud_idx, 'sales_t'] = data.loc[fraud_idx, 'sales_t'] * 0.7

# Inflate COGS (shrinking margins)
data.loc[fraud_idx, 'COGS_t'] = data.loc[fraud_idx, 'COGS_t'] * 1.4

# Recalculate ratios
data['DSRI'] = (data['receivables_t'] / data['sales_t']) / (data['receivables_t1'] / data['sales_t1'])
data['GMI'] = ((data['sales_t1'] - data['COGS_t1']) / data['sales_t1']) / ((data['sales_t'] - data['COGS_t']) / data['sales_t'])
data['AQI'] = (1 - (data['PPE_t'] + data['receivables_t']) / data['total_assets_t']) / (1 - (data['PPE_t1'] + data['receivables_t1']) / data['total_assets_t1'])
data['SGI'] = data['sales_t'] / data['sales_t1']

print(data[['company_id', 'DSRI', 'GMI', 'AQI', 'SGI', 'fraud']].head(10))
print(data[['company_id', 'DSRI', 'GMI', 'AQI', 'SGI', 'fraud']].tail(10))


   company_id       DSRI       GMI       AQI    SGI  fraud
0           1  45.553936 -0.105263 -17.28125  0.343      1
1           2  45.553936 -0.105263 -17.28125  0.343      1
2           3  45.553936 -0.105263 -17.28125  0.343      1
3           4  45.553936 -0.105263 -17.28125  0.343      1
4           5  45.553936 -0.105263 -17.28125  0.343      1
5           6  45.553936 -0.105263 -17.28125  0.343      1
6           7  45.553936 -0.105263 -17.28125  0.343      1
7           8  45.553936 -0.105263 -17.28125  0.343      1
8           9  45.553936 -0.105263 -17.28125  0.343      1
9          10  45.553936 -0.105263 -17.28125  0.343      1
    company_id  DSRI  GMI  AQI  SGI  fraud
90          91   1.0  1.0  1.0  1.0      0
91          92   1.0  1.0  1.0  1.0      0
92          93   1.0  1.0  1.0  1.0      0
93          94   1.0  1.0  1.0  1.0      0
94          95   1.0  1.0  1.0  1.0      0
95          96   1.0  1.0  1.0  1.0      0
96          97   1.0  1.0  1.0  1.0      0
97     

In [8]:
# Beneish Index 5: DEPI - Depreciation Index
# Measures whether depreciation rate is slowing down (assets being stretched)
data['DEPI'] = (data['PPE_t1'] / (data['PPE_t1'] + data['PPE_t'])) / (data['PPE_t'] / (data['PPE_t'] + data['PPE_t1']))

# Beneish Index 6: SGAI - Sales General and Administrative Expenses Index
# We approximate SGA as 10% of sales
data['SGA_t'] = data['sales_t'] * 0.10
data['SGA_t1'] = data['sales_t1'] * 0.10
data['SGAI'] = (data['SGA_t'] / data['sales_t']) / (data['SGA_t1'] / data['sales_t1'])

# Beneish Index 7: LVGI - Leverage Index
# Measures whether debt levels are increasing
data['LVGI'] = (data['total_debt_t'] / data['total_assets_t']) / (data['total_debt_t'] / data['total_assets_t1'])

# Beneish Index 8: TATA - Total Accruals to Total Assets
# Measures gap between reported income and cash flow (accruals manipulation)
data['TATA'] = (data['net_income_t'] - data['cash_flow_ops_t']) / data['total_assets_t']

print(data[['company_id', 'DEPI', 'SGAI', 'LVGI', 'TATA', 'fraud']].head(10))

   company_id  DEPI  SGAI  LVGI      TATA  fraud
0           1   1.0   1.0   1.0 -0.257909      1
1           2   1.0   1.0   1.0  0.232791      1
2           3   1.0   1.0   1.0  0.382870      1
3           4   1.0   1.0   1.0  0.183877      1
4           5   1.0   1.0   1.0  0.143063      1
5           6   1.0   1.0   1.0 -0.050231      1
6           7   1.0   1.0   1.0  0.004042      1
7           8   1.0   1.0   1.0  0.181482      1
8           9   1.0   1.0   1.0 -0.010781      1
9          10   1.0   1.0   1.0 -0.228465      1


In [10]:
# Calculate Beneish M-Score
# These weights come from Beneish's original 1999 research paper
data['M_Score'] = (
    -4.84
    + (0.920 * data['DSRI'])
    + (0.528 * data['GMI'])
    + (0.404 * data['AQI'])
    + (0.892 * data['SGI'])
    + (0.115 * data['DEPI'])
    - (0.172 * data['SGAI'])
    + (4.679 * data['TATA'])
    - (0.327 * data['LVGI'])
)

print(data[['company_id', 'M_Score', 'fraud']].head(20))

print(data[['company_id', 'M_Score', 'fraud']].tail(10))

    company_id    M_Score  fraud
0            1  28.747619      1
1            2  31.043604      1
2            3  31.745822      1
3            4  30.814732      1
4            5  30.623765      1
5            6  29.719343      1
6            7  29.973283      1
7            8  30.803527      1
8            9  29.903927      1
9           10  28.885386      1
10          11  29.522795      1
11          12  31.083539      1
12          13  31.597619      1
13          14  28.363877      1
14          15  29.979414      1
15          16  31.437688      1
16          17  30.161126      1
17          18  29.717385      1
18          19  30.503873      1
19          20  29.450486      1
    company_id   M_Score  fraud
90          91 -1.330773      0
91          92 -1.391576      0
92          93 -2.395365      0
93          94 -2.852212      0
94          95 -3.679989      0
95          96 -2.529738      0
96          97 -2.368398      0
97          98 -2.307618      0
98          99 -2.1